# Quantitative Strategy Notebook — Cell Fracture Edition

**Objective:** Build a high-performance, modular backtesting framework for FTMO-style assets using an ensemble of volatility-normalized indicators, regime filters, and strict out-of-sample validation.

**Tech Stack:** `yfinance` · `TA-Lib` · `vectorbt` · `scipy.stats` · `pandas`

**Protocol:** Every logical step occupies its own cell, preceded by a markdown explanation. Every data transformation is followed by a print-based audit trail. No black boxes.

**Sources & Methodology:**
- Robert Carver, *Systematic Trading* — forecast scaling, volatility normalization, out-of-sample testing
- Giordano (2018 Dow Award) — ATR Trend/Breakout System, Ranking Model, GARCH volatility
- Keller & van Putten — Flexible Asset Allocation momentum factors
- Zarattini & Antonacci (2025 Dow Award) — Keltner Channels, industry trend-following

---


## 1. Environment & Dependencies

### What: Install and import all required libraries.
### Why: A single cell for environment setup prevents scattered imports.
### Assumption: Running in a Jupyter/Colab environment with pip access.


In [ ]:
# !pip install yfinance TA-Lib vectorbt scipy pandas numpy matplotlib --quiet

import yfinance as yf
import talib
import numpy as np
import pandas as pd
import vectorbt as vbt
import warnings
from scipy import stats
import matplotlib.pyplot as plt
from itertools import product

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 120)

print("✅ All imports successful")
print(f"   vectorbt version: {vbt.__version__}")
print(f"   pandas version:   {pd.__version__}")
print(f"   numpy version:    {np.__version__}")


## 2. Configuration Cell — Single Source of Truth

### What: Define all hyperparameters, tickers, date ranges, and split ratios in one place.
### Why: Prevents "magic numbers" buried in code. Any parameter change happens HERE.
### Assumption: BTC-USD trades 365 days/year (no weekday gaps). Equity/FX tickers have weekday gaps.


In [ ]:
# ═══════════════════════════════════════════════════════════════
# CONFIGURATION — EDIT ONLY THIS CELL
# ═══════════════════════════════════════════════════════════════

TICKER       = "BTC-USD"           # Asset to backtest
START_DATE   = "2018-01-01"        # Download start
END_DATE     = None                # None = today
TRAIN_RATIO  = 0.70                # 70% training, 30% OOS

# Indicator Grid Ranges (for optimization)
EMA1_RANGE   = range(5, 25, 2)     # Fast EMA
EMA2_RANGE   = range(20, 60, 5)    # Medium EMA
EMA3_RANGE   = range(50, 120, 10)  # Slow EMA

# Donchian Channel
DC_RANGE     = range(10, 50, 5)    # Donchian lookback

# MACD-V (Volatility-Normalized)
MACDV_FAST   = range(8, 16, 2)
MACDV_SLOW   = range(20, 35, 3)
MACDV_SIGNAL = range(5, 12, 2)
ATR_LEN      = 14                  # For normalization

# Regime & Volatility Filters
SMA_REGIME_LEN   = 200             # 200-day SMA regime filter
VOL_LOOKBACK     = 20              # ATR lookback for volatility filter
VOL_PERCENTILE_LO = 5              # Below 5th percentile = dead market
VOL_PERCENTILE_HI = 99             # Above 99th percentile = extreme

# Backtest Settings
BATCH_SIZE   = 1000                # Param combos per vectorbt batch
SIGNAL_SHIFT = 1                   # Bars to shift signal before execution (anti-lookahead)

# Sensitivity Analysis
SENSITIVITY_RANGE = 5              # +/- N steps around optimal

print("✅ Configuration loaded")
print(f"   Ticker: {TICKER}")
print(f"   Date range: {START_DATE} → {END_DATE or 'today'}")
print(f"   Train/OOS split: {TRAIN_RATIO:.0%} / {1-TRAIN_RATIO:.0%}")
print(f"   Signal shift: {SIGNAL_SHIFT} bar(s) — anti-lookahead")


## 3. Data Acquisition — yfinance Downloader

### What: Download OHLCV data and handle yfinance's MultiIndex column format.
### Why: yfinance returns MultiIndex columns `(Price, Ticker)` which breaks naive `df['Close']` access.
### Assumption: yfinance auto_adjust=True (default). No dividend/split adjustments needed for crypto.


In [ ]:
# Download OHLCV data
raw_df = yf.download(TICKER, start=START_DATE, end=END_DATE)

print(f"\n📥 Downloaded {len(raw_df)} rows for {TICKER}")
print(f"   Shape: {raw_df.shape}")
print(f"   Columns type: {type(raw_df.columns).__name__}")
print(f"   Date range: {raw_df.index[0].date()} → {raw_df.index[-1].date()}")
print(f"\n   First 3 rows:")
print(raw_df.head(3))


## 4. Multi-Index Column Flattening

### What: Detect and flatten yfinance's MultiIndex columns to simple strings.
### Why: `df.columns = pd.MultiIndex([('Close','BTC-USD'), ...])` causes downstream failures in TA-Lib and vectorbt which expect flat column names.
### Assumption: Single-ticker download produces a 2-level MultiIndex `(Price, Ticker)`.

> **Skeptic Note:** If yfinance changes its API (again), this cell is the first to break.


In [ ]:
# Handle MultiIndex columns from yfinance
if isinstance(raw_df.columns, pd.MultiIndex):
    print("⚠️  MultiIndex detected — flattening to level 0 (Price names)")
    df = raw_df.droplevel('Ticker', axis=1)
else:
    print("✅ Columns are already flat")
    df = raw_df.copy()

# Ensure standard column names exist
required_cols = ['Open', 'High', 'Low', 'Close', 'Volume']
missing = [c for c in required_cols if c not in df.columns]
assert len(missing) == 0, f"❌ Missing columns: {missing}"

print(f"\n✅ Flattened DataFrame")
print(f"   Shape: {df.shape}")
print(f"   Columns: {list(df.columns)}")
print(f"   Dtypes:\n{df.dtypes}")


## 5. Data Sanity Audit — Timestamps, NaNs, and Monotonicity

### What: Verify the data is strictly time-ordered, check for NaN holes, and detect duplicate indices.
### Why: Non-monotonic timestamps or hidden NaN patches will silently corrupt every indicator downstream. Per Carver: "beware overconfidence" in backtest results built on unverified data.
### Assumption: Daily data with no intraday bars mixed in.


In [ ]:
# ─── 5a. Monotonic Index Check ───
is_monotonic = df.index.is_monotonic_increasing
is_unique    = df.index.is_unique
print(f"🔍 Index Audit:")
print(f"   Monotonically increasing: {is_monotonic}")
print(f"   All dates unique:         {is_unique}")
assert is_monotonic, "❌ CRITICAL: Index is NOT time-ordered!"
assert is_unique,    "❌ CRITICAL: Duplicate dates found!"


In [ ]:
# ─── 5b. NaN Audit ───
nan_counts = df.isna().sum()
nan_pct    = (df.isna().sum() / len(df) * 100).round(2)
print(f"\n🔍 NaN Audit:")
print(f"   Total rows: {len(df)}")
for col in df.columns:
    print(f"   {col:>8}: {nan_counts[col]:>5} NaNs ({nan_pct[col]:.2f}%)")

# Forward-fill small gaps (weekends for crypto shouldn't exist, but safety net)
initial_nans = df.isna().sum().sum()
df = df.ffill().bfill()
final_nans = df.isna().sum().sum()
print(f"\n   NaNs before fill: {initial_nans}")
print(f"   NaNs after fill:  {final_nans}")
assert final_nans == 0, "❌ NaNs remain after fill!"


In [ ]:
# ─── 5c. Target Contamination Check ───
# Verify no future data leaks into features. At this stage, we only have raw OHLCV.
# The risk comes later when we compute indicators — flag it here as a checkpoint.

print("🔍 Target Contamination Pre-Check:")
print("   ✅ Raw OHLCV only — no derived features yet")
print("   ⚠️  REMINDER: All indicators MUST use .shift(SIGNAL_SHIFT) before execution")
print(f"   ⚠️  SIGNAL_SHIFT = {SIGNAL_SHIFT} bar(s)")
print(f"\n   Data ready: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"   Date range: {df.index[0].date()} → {df.index[-1].date()}")


## 6. Extract Core Price Series

### What: Pull out Close, High, Low as standalone Series for indicator calculations.
### Why: TA-Lib and vectorbt expect 1D arrays/Series, not DataFrames. Extracting here prevents repeated indexing.
### Assumption: All series are aligned on the same DatetimeIndex.


In [ ]:
close = df['Close'].astype(float)
high  = df['High'].astype(float)
low   = df['Low'].astype(float)
volume = df['Volume'].astype(float)

print(f"✅ Price series extracted")
print(f"   close:  {close.shape}, dtype={close.dtype}, first={close.iloc[0]:.2f}, last={close.iloc[-1]:.2f}")
print(f"   high:   {high.shape},  dtype={high.dtype}")
print(f"   low:    {low.shape},   dtype={low.dtype}")
print(f"   volume: {volume.shape}, dtype={volume.dtype}")

# Sanity: High >= Close >= Low on every bar
violations = ((high < close) | (close < low)).sum()
print(f"\n   OHLC integrity (High >= Close >= Low): {len(close) - violations}/{len(close)} bars pass")
if violations > 0:
    print(f"   ⚠️  {violations} bars have OHLC violations — possible bad data")


## 7. Vectorized Indicator Engine — The Ensemble

We compute three indicator families, each representing a different "lens" on the market:

| Indicator | What It Measures | Normalization |
|-----------|-----------------|---------------|
| **Triple EMA** | Trend direction via crossover hierarchy | Raw crossover (binary) |
| **Donchian Channel** | Breakout above/below N-bar high/low | Price-based |
| **MACD-V** | Momentum normalized by volatility (ATR) | "Invariant Ruler" — comparable across regimes |

The MACD-V concept follows Carver's principle: *"forecast values should be roughly similar across instruments and long periods of time, otherwise your trading rule could be badly specified and not properly volatility normalised."*


### 7a. ATR Calculation (Shared Denominator)

### What: Compute Average True Range — the volatility "ruler" used to normalize MACD-V.
### Why: Per Giordano (2018 Dow Award), ATR captures "periods of great and low directionality" and serves as the volatility band for trend identification.
### Assumption: ATR_LEN=14 is the Wilder default. Changing this changes ALL downstream normalization.


In [ ]:
# ATR — the volatility ruler
atr = pd.Series(
    talib.ATR(high.values, low.values, close.values, timeperiod=ATR_LEN),
    index=close.index
)

print(f"✅ ATR({ATR_LEN}) computed")
print(f"   Shape: {atr.shape}")
print(f"   NaNs:  {atr.isna().sum()} (expected: {ATR_LEN} warmup rows)")
print(f"   Mean:  {atr.mean():.2f}")
print(f"   Last:  {atr.iloc[-1]:.2f}")


### 7b. Triple EMA Signal Generator

### What: Compute three EMAs and generate entry/exit signals from crossover hierarchy.
### Why: Triple EMA is a momentum filter — EMA1 > EMA2 > EMA3 means aligned bullish trend.
### Assumption: Entry = fast crosses above both medium and slow. Exit = fast crosses below medium.


In [ ]:
def compute_3ema_signals(close, ema1_len, ema2_len, ema3_len):
    """Compute Triple EMA entry/exit signals.
    
    Entry: EMA1 > EMA2 AND EMA2 > EMA3 (alignment)
    Exit:  EMA1 < EMA2 (fast loses momentum)
    
    Returns boolean Series (entries, exits) — NOT shifted yet.
    """
    ema1 = pd.Series(talib.EMA(close.values, timeperiod=ema1_len), index=close.index)
    ema2 = pd.Series(talib.EMA(close.values, timeperiod=ema2_len), index=close.index)
    ema3 = pd.Series(talib.EMA(close.values, timeperiod=ema3_len), index=close.index)
    
    # Aligned bullish: fast > medium > slow
    bullish = (ema1 > ema2) & (ema2 > ema3)
    
    # Entry: transition INTO aligned state
    entries = bullish & ~bullish.shift(1).fillna(False)
    
    # Exit: fast crosses below medium
    exits = (ema1 < ema2) & (ema1.shift(1) >= ema2.shift(1))
    
    return entries, exits

# Demo with middle-of-range params
demo_ent, demo_exi = compute_3ema_signals(close, 10, 30, 70)
print(f"✅ 3EMA signal generator defined")
print(f"   Demo (10/30/70): {demo_ent.sum()} entries, {demo_exi.sum()} exits over {len(close)} bars")


### 7c. Donchian Channel Signal Generator

### What: Compute Donchian Channels (N-bar high/low) and generate breakout signals.
### Why: Donchian breakouts are the original trend-following signal (Turtle Traders). They capture pure price extremes without any smoothing lag.
### Assumption: Entry = close breaks above N-bar high. Exit = close breaks below N-bar low.


In [ ]:
def compute_donchian_signals(close, high, low, dc_len):
    """Compute Donchian Channel breakout signals.
    
    Upper Band = highest high of last dc_len bars
    Lower Band = lowest low of last dc_len bars
    Entry: close > upper band (shifted 1 to avoid look-ahead)
    Exit:  close < lower band (shifted 1)
    """
    upper = high.rolling(dc_len).max().shift(1)  # Previous N-bar high
    lower = low.rolling(dc_len).min().shift(1)   # Previous N-bar low
    
    entries = (close > upper) & ~(close.shift(1) > upper.shift(1))
    exits   = (close < lower) & ~(close.shift(1) < lower.shift(1))
    
    return entries, exits

demo_dc_ent, demo_dc_exi = compute_donchian_signals(close, high, low, 20)
print(f"✅ Donchian Channel signal generator defined")
print(f"   Demo (DC-20): {demo_dc_ent.sum()} entries, {demo_dc_exi.sum()} exits")


### 7d. MACD-V: Volatility-Normalized MACD (The "Invariant Ruler")

### What: Standard MACD line divided by ATR to produce a volatility-normalized momentum score.
### Why: Raw MACD values are meaningless across assets — $100 MACD on BTC vs $0.001 on EURUSD. Dividing by ATR creates an "invariant ruler" comparable across instruments and regimes. This follows Carver's forecast scaling principle and the Keltner Channel logic from Zarattini (2025 Dow Award).
### Assumption: MACD-V > +1.0 = strong bullish momentum (1 ATR unit above zero). Entry/exit at zero-line cross.


In [ ]:
def compute_macdv_signals(close, atr, fast_len, slow_len, signal_len):
    """Compute MACD-V (Volatility-Normalized MACD).
    
    MACD-V = (EMA_fast - EMA_slow) / ATR
    Signal = EMA(MACD-V, signal_len)
    
    Entry: MACD-V crosses above Signal
    Exit:  MACD-V crosses below Signal
    """
    macd_line = pd.Series(
        talib.EMA(close.values, timeperiod=fast_len) - talib.EMA(close.values, timeperiod=slow_len),
        index=close.index
    )
    
    # Normalize by ATR — the "invariant ruler"
    macd_v = macd_line / atr.replace(0, np.nan)
    
    signal_line = pd.Series(
        talib.EMA(macd_v.values, timeperiod=signal_len),
        index=close.index
    )
    
    entries = (macd_v > signal_line) & (macd_v.shift(1) <= signal_line.shift(1))
    exits   = (macd_v < signal_line) & (macd_v.shift(1) >= signal_line.shift(1))
    
    return entries, exits, macd_v, signal_line

demo_mv_ent, demo_mv_exi, demo_macdv, demo_sig = compute_macdv_signals(close, atr, 12, 26, 9)
print(f"✅ MACD-V signal generator defined")
print(f"   Demo (12/26/9): {demo_mv_ent.sum()} entries, {demo_mv_exi.sum()} exits")
print(f"   MACD-V range: [{demo_macdv.min():.3f}, {demo_macdv.max():.3f}]")
print(f"   MACD-V mean:  {demo_macdv.mean():.3f} (should be near 0)")


## 8. Regime Filter — 200-Day SMA Gate

### What: Only allow LONG entries when price is above the 200-day SMA (bullish regime).
### Why: Per Carver and Faber, trend-following works best when aligned with the dominant regime. Trading longs in a bear market is a documented way to blow up FTMO accounts.
### Assumption: SMA_REGIME_LEN=200 is standard. This is a FILTER, not a signal — it gates other signals.

> **Skeptic Note:** The 200-SMA is a lagging indicator by definition. It will miss the first ~200 bars of any new trend. This is a feature, not a bug — it reduces false starts.


In [ ]:
sma_200 = pd.Series(
    talib.SMA(close.values, timeperiod=SMA_REGIME_LEN),
    index=close.index
)
regime_bullish = close > sma_200

print(f"✅ Regime Filter (SMA-{SMA_REGIME_LEN}) computed")
print(f"   NaNs (warmup):    {sma_200.isna().sum()}")
print(f"   Bullish bars:     {regime_bullish.sum()} ({regime_bullish.mean():.1%})")
print(f"   Bearish bars:     {(~regime_bullish).sum()} ({(~regime_bullish).mean():.1%})")
print(f"   Current regime:   {'BULLISH ✅' if regime_bullish.iloc[-1] else 'BEARISH ❌'}")


## 9. Volatility Filter — Dead Market & Extreme Regime Exclusion

### What: Exclude entries when ATR is below the 5th percentile (dead market) or above the 99th percentile (extreme crisis).
### Why: Low volatility means no edge for momentum strategies. Extreme volatility means spreads blow out and slippage becomes catastrophic. Per Giordano: "volatility measures the deviations… but is blind compared to the trend."
### Assumption: Percentiles are computed on a rolling 252-day window to be adaptive, not on the full dataset (which would leak future info).


In [ ]:
# Rolling percentile rank of ATR (252-day window)
atr_pctile = atr.rolling(252).apply(lambda x: stats.percentileofscore(x, x.iloc[-1]), raw=False)

vol_ok = (atr_pctile > VOL_PERCENTILE_LO) & (atr_pctile < VOL_PERCENTILE_HI)

print(f"✅ Volatility Filter computed (rolling 252-day percentile)")
print(f"   Excluded (too low, <{VOL_PERCENTILE_LO}th pct):  {(atr_pctile <= VOL_PERCENTILE_LO).sum()} bars")
print(f"   Excluded (extreme, >{VOL_PERCENTILE_HI}th pct): {(atr_pctile >= VOL_PERCENTILE_HI).sum()} bars")
print(f"   Tradeable bars:   {vol_ok.sum()} ({vol_ok.mean():.1%})")
print(f"   NaNs (warmup):    {vol_ok.isna().sum()}")

# Fill NaN warmup period as False (don't trade during warmup)
vol_ok = vol_ok.fillna(False)


## 10. Time-Ordered Train/Validation Split

### What: Split data into Training (first 70%) and Out-of-Sample (last 30%) by time order.
### Why: Random splits are ILLEGAL in time series. Per Carver: "In sample testing should be avoided as it will produce extremely optimistic back-test results." We use a strict temporal split — train on the past, test on the future.
### Assumption: TRAIN_RATIO=0.70. No overlap, no shuffling, no peeking.

> **Critical:** ALL optimization happens on train data only. OOS is touched ONCE at the end.


In [ ]:
split_idx = int(len(df) * TRAIN_RATIO)
split_date = df.index[split_idx]

train_close = close.iloc[:split_idx]
test_close  = close.iloc[split_idx:]
train_high  = high.iloc[:split_idx]
test_high   = high.iloc[split_idx:]
train_low   = low.iloc[:split_idx]
test_low    = low.iloc[split_idx:]
train_atr   = atr.iloc[:split_idx]
test_atr    = atr.iloc[split_idx:]
train_regime = regime_bullish.iloc[:split_idx]
test_regime  = regime_bullish.iloc[split_idx:]
train_vol_ok = vol_ok.iloc[:split_idx]
test_vol_ok  = vol_ok.iloc[split_idx:]

print(f"✅ Time-Ordered Split at index {split_idx} ({split_date.date()})")
print(f"   TRAIN: {len(train_close)} bars | {train_close.index[0].date()} → {train_close.index[-1].date()}")
print(f"   OOS:   {len(test_close)} bars  | {test_close.index[0].date()} → {test_close.index[-1].date()}")
print(f"   Ratio: {len(train_close)/len(close):.1%} / {len(test_close)/len(close):.1%}")
print(f"\n   ⚠️  OOS data will NOT be touched until Section 13.")


## 11. Vectorized Grid Search — Triple EMA Optimization

### What: Exhaustively test all EMA1×EMA2×EMA3 parameter combinations on TRAINING data only, using vectorbt for high-speed vectorized backtesting.
### Why: Grid search finds the Sharpe-maximizing parameters. We batch in groups of BATCH_SIZE to prevent OOM crashes.
### Assumption: Sharpe Ratio is the objective (risk-adjusted, not raw return). All signals are shifted by SIGNAL_SHIFT bars to prevent lookahead.

> **Anti-Lookahead:** `entries.shift(SIGNAL_SHIFT)` ensures the signal generated on bar T is executed on bar T+1 (or later). This is the single most important line in the notebook.


In [ ]:
# Generate all valid parameter combinations (ema1 < ema2 < ema3)
all_combos = [
    (e1, e2, e3) 
    for e1 in EMA1_RANGE 
    for e2 in EMA2_RANGE 
    for e3 in EMA3_RANGE 
    if e1 < e2 < e3
]

print(f"✅ Parameter grid generated")
print(f"   Total valid combos: {len(all_combos)}")
print(f"   EMA1 range: {list(EMA1_RANGE)}")
print(f"   EMA2 range: {list(EMA2_RANGE)}")
print(f"   EMA3 range: {list(EMA3_RANGE)}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Estimated batches: {len(all_combos) // BATCH_SIZE + 1}")


### 11a. Batched Grid Search Execution

### What: Run the actual optimization loop, processing BATCH_SIZE combos at a time.
### Why: vectorbt can handle thousands of parallel backtests but will OOM if given too many at once.
### Assumption: Using `.sharpe_ratio(freq='D')` — annualized Sharpe assuming daily data.


In [ ]:
results_list = []
n_batches = (len(all_combos) + BATCH_SIZE - 1) // BATCH_SIZE

print(f"🔄 Starting batched grid search ({n_batches} batches)...\n")

for batch_idx in range(n_batches):
    batch_start = batch_idx * BATCH_SIZE
    batch_end   = min(batch_start + BATCH_SIZE, len(all_combos))
    batch       = all_combos[batch_start:batch_end]
    
    for (e1, e2, e3) in batch:
        try:
            ent, exi = compute_3ema_signals(train_close, e1, e2, e3)
            
            # Apply regime + volatility filters
            ent = ent & train_regime & train_vol_ok
            
            # ANTI-LOOKAHEAD: shift signals forward
            ent = ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
            exi = exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
            
            if ent.sum() < 5:  # Need minimum trades for valid Sharpe
                continue
            
            pf = vbt.Portfolio.from_signals(
                train_close, entries=ent, exits=exi,
                init_cash=100_000, fees=0.001, freq='D'
            )
            
            sr = pf.sharpe_ratio()
            if np.isnan(sr):
                continue
                
            results_list.append({
                'ema1': e1, 'ema2': e2, 'ema3': e3,
                'sharpe': float(sr),
                'total_return': float(pf.total_return()),
                'max_dd': float(pf.max_drawdown()),
                'n_trades': int(pf.trades.count()),
            })
        except Exception:
            pass
    
    if (batch_idx + 1) % 5 == 0 or batch_idx == n_batches - 1:
        print(f"   Batch {batch_idx+1}/{n_batches} complete — {len(results_list)} valid results so far")

print(f"\n✅ Grid search complete: {len(results_list)} valid parameter sets evaluated")


### 11b. Optimization Results — Top Parameters

### What: Rank all results by Sharpe Ratio and extract the best parameters.
### Why: We maximize risk-adjusted returns, not raw returns. Per Carver: "a Sharpe ratio after costs of 0.53" is a realistic target for diversified systematic strategies.
### Assumption: The top result is not necessarily robust — that's what sensitivity analysis (Section 14) is for.


In [ ]:
if not results_list:
    raise RuntimeError("❌ No valid results! Check data or parameter ranges.")

results_df = pd.DataFrame(results_list).sort_values('sharpe', ascending=False).reset_index(drop=True)

# Best parameters
best = results_df.iloc[0]
BEST_EMA1 = int(best['ema1'])
BEST_EMA2 = int(best['ema2'])
BEST_EMA3 = int(best['ema3'])

print(f"🏆 TOP 10 PARAMETER SETS (by Sharpe Ratio)")
print("=" * 80)
print(results_df.head(10).to_string(index=False))
print(f"\n{'='*80}")
print(f"🏆 BEST: EMA({BEST_EMA1}, {BEST_EMA2}, {BEST_EMA3})")
print(f"   Sharpe:       {best['sharpe']:.4f}")
print(f"   Total Return: {best['total_return']:.2%}")
print(f"   Max Drawdown: {best['max_dd']:.2%}")
print(f"   # Trades:     {best['n_trades']}")
print(f"\n   ⚠️  This is IN-SAMPLE. Real performance = OOS (Section 13).")


## 12. In-Sample Backtest — Full Equity Curve

### What: Run the best parameters on training data and plot the equity curve.
### Why: Visual inspection catches things metrics miss — suspicious flatlines, single-trade dominance, or regime-dependent performance.
### Assumption: Using the BEST parameters from optimization. This IS still in-sample.


In [ ]:
# Generate signals with best parameters on TRAINING data
is_entries, is_exits = compute_3ema_signals(train_close, BEST_EMA1, BEST_EMA2, BEST_EMA3)

# Apply filters
is_entries = is_entries & train_regime & train_vol_ok

# Anti-lookahead shift
is_entries = is_entries.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
is_exits   = is_exits.shift(SIGNAL_SHIFT).fillna(False).astype(bool)

# Run backtest
pf_is = vbt.Portfolio.from_signals(
    train_close, entries=is_entries, exits=is_exits,
    init_cash=100_000, fees=0.001, freq='D'
)

print(f"✅ In-Sample Backtest — EMA({BEST_EMA1}, {BEST_EMA2}, {BEST_EMA3})")
print(f"{'='*60}")
print(f"   Sharpe Ratio:  {pf_is.sharpe_ratio():.4f}")
print(f"   Sortino Ratio: {pf_is.sortino_ratio():.4f}")
print(f"   Total Return:  {pf_is.total_return():.2%}")
print(f"   Max Drawdown:  {pf_is.max_drawdown():.2%}")
print(f"   # Trades:      {pf_is.trades.count()}")
print(f"   Win Rate:      {pf_is.trades.win_rate():.2%}")


In [ ]:
# Plot In-Sample equity curve
fig, axes = plt.subplots(2, 1, figsize=(14, 8), gridspec_kw={'height_ratios': [3, 1]})
fig.suptitle(f'In-Sample Equity Curve — EMA({BEST_EMA1},{BEST_EMA2},{BEST_EMA3})', fontsize=14, fontweight='bold')

# Equity
pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].set_title(f'Sharpe: {pf_is.sharpe_ratio():.3f} | Return: {pf_is.total_return():.1%} | MaxDD: {pf_is.max_drawdown():.1%}')
axes[0].grid(alpha=0.3)

# Drawdown
pf_is.drawdown().plot(ax=axes[1], color='crimson', linewidth=1)
axes[1].fill_between(pf_is.drawdown().index, pf_is.drawdown().values, alpha=0.3, color='crimson')
axes[1].set_ylabel('Drawdown')
axes[1].set_xlabel('Date')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 13. Out-of-Sample Validation — The Moment of Truth

### What: Apply the EXACT same parameters found in-sample to the UNTOUCHED validation data.
### Why: This is the ONLY honest measure of strategy performance. Per Carver: "Use the data from the first half to fit the best variation. Then use that single variation to test your performance on the second out of sample period."
### Assumption: Parameters are FROZEN from training. No re-optimization allowed.

> **Critical:** If OOS Sharpe is dramatically worse than IS Sharpe, the strategy is overfit. Period.


In [ ]:
# Generate signals on OOS data with FROZEN parameters
oos_entries, oos_exits = compute_3ema_signals(test_close, BEST_EMA1, BEST_EMA2, BEST_EMA3)

# Apply filters (computed on OOS data)
oos_entries = oos_entries & test_regime & test_vol_ok

# Anti-lookahead shift
oos_entries = oos_entries.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
oos_exits   = oos_exits.shift(SIGNAL_SHIFT).fillna(False).astype(bool)

# Run OOS backtest
pf_oos = vbt.Portfolio.from_signals(
    test_close, entries=oos_entries, exits=oos_exits,
    init_cash=100_000, fees=0.001, freq='D'
)

print(f"✅ Out-of-Sample Backtest — EMA({BEST_EMA1}, {BEST_EMA2}, {BEST_EMA3})")
print(f"{'='*60}")
print(f"   Sharpe Ratio:  {pf_oos.sharpe_ratio():.4f}")
print(f"   Sortino Ratio: {pf_oos.sortino_ratio():.4f}")
print(f"   Total Return:  {pf_oos.total_return():.2%}")
print(f"   Max Drawdown:  {pf_oos.max_drawdown():.2%}")
print(f"   # Trades:      {pf_oos.trades.count()}")
oos_wr = pf_oos.trades.win_rate() if pf_oos.trades.count() > 0 else 0
print(f"   Win Rate:      {oos_wr:.2%}")


### 13a. IS vs OOS Comparison Table

### What: Side-by-side comparison of In-Sample vs Out-of-Sample metrics.
### Why: Sharpe degradation > 50% is a strong overfitting signal. Drawdown expansion in OOS is a red flag.
### Assumption: Healthy degradation is 20-40%. Anything above 50% = likely overfit.


In [ ]:
# Build comparison table
is_sharpe  = pf_is.sharpe_ratio()
oos_sharpe = pf_oos.sharpe_ratio()
sharpe_degrad = (1 - oos_sharpe / is_sharpe) * 100 if is_sharpe != 0 else float('inf')

comparison = pd.DataFrame({
    'Metric': ['Sharpe Ratio', 'Sortino Ratio', 'Total Return', 'Max Drawdown', 
               'Trades', 'Win Rate'],
    'In-Sample': [
        f"{pf_is.sharpe_ratio():.4f}",
        f"{pf_is.sortino_ratio():.4f}",
        f"{pf_is.total_return():.2%}",
        f"{pf_is.max_drawdown():.2%}",
        f"{pf_is.trades.count()}",
        f"{pf_is.trades.win_rate():.2%}",
    ],
    'Out-of-Sample': [
        f"{pf_oos.sharpe_ratio():.4f}",
        f"{pf_oos.sortino_ratio():.4f}",
        f"{pf_oos.total_return():.2%}",
        f"{pf_oos.max_drawdown():.2%}",
        f"{pf_oos.trades.count()}",
        f"{oos_wr:.2%}",
    ],
})

print("📊 IN-SAMPLE vs OUT-OF-SAMPLE COMPARISON")
print("=" * 70)
print(comparison.to_string(index=False))
print(f"\n{'='*70}")
print(f"   Sharpe Degradation: {sharpe_degrad:.1f}%", end="")
if abs(sharpe_degrad) < 30:
    print(" ✅ (Healthy)")
elif abs(sharpe_degrad) < 50:
    print(" ⚠️ (Moderate — proceed with caution)")
else:
    print(" ❌ (Severe — likely overfit!)")


In [ ]:
# Plot OOS equity curve alongside IS
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('In-Sample vs Out-of-Sample Equity Curves', fontsize=14, fontweight='bold')

pf_is.value().plot(ax=axes[0], color='navy', linewidth=1.5)
axes[0].set_title(f'In-Sample | Sharpe: {pf_is.sharpe_ratio():.3f}')
axes[0].set_ylabel('Portfolio Value ($)')
axes[0].grid(alpha=0.3)

pf_oos.value().plot(ax=axes[1], color='darkgreen', linewidth=1.5)
axes[1].set_title(f'Out-of-Sample | Sharpe: {pf_oos.sharpe_ratio():.3f}')
axes[1].set_ylabel('Portfolio Value ($)')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()


## 14. Sensitivity Analysis — Are We on a Peak or a Plateau?

### What: Vary each parameter independently around the optimum and measure Sharpe degradation.
### Why: If performance collapses when EMA1 changes by ±2, you're sitting on a "lucky" spike, not a robust parameter region. Per Carver: "favour more complex rules that fit the data better but won't do as well in real trading."
### Assumption: We vary each EMA independently while holding the others at their optimal values.

> **What to look for:** A flat "plateau" around the optimum = robust. A sharp "needle" = fragile overfit.


In [ ]:
def run_sensitivity(base_e1, base_e2, base_e3, n_range=SENSITIVITY_RANGE):
    """Run sensitivity analysis around base parameters."""
    
    def eval_combo(e1, e2, e3, data_close, data_regime, data_vol):
        ent, exi = compute_3ema_signals(data_close, e1, e2, e3)
        ent = ent & data_regime & data_vol
        ent = ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        exi = exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
        
        if ent.sum() < 3:
            return np.nan, np.nan
        
        pf = vbt.Portfolio.from_signals(
            data_close, entries=ent, exits=exi,
            init_cash=100_000, fees=0.001, freq='D'
        )
        return float(pf.sharpe_ratio()), float(pf.max_drawdown())
    
    rows = []
    
    # Vary EMA1
    for delta in range(-n_range, n_range + 1):
        e1 = base_e1 + delta * 2
        if e1 < 3 or e1 >= base_e2:
            continue
        is_sr, is_dd = eval_combo(e1, base_e2, base_e3, train_close, train_regime, train_vol_ok)
        oos_sr, oos_dd = eval_combo(e1, base_e2, base_e3, test_close, test_regime, test_vol_ok)
        rows.append({'param': 'EMA1', 'value': e1, 'is_sharpe': is_sr, 'oos_sharpe': oos_sr, 'is_dd': is_dd, 'oos_dd': oos_dd})
    
    # Vary EMA2
    for delta in range(-n_range, n_range + 1):
        e2 = base_e2 + delta * 5
        if e2 <= base_e1 or e2 >= base_e3:
            continue
        is_sr, is_dd = eval_combo(base_e1, e2, base_e3, train_close, train_regime, train_vol_ok)
        oos_sr, oos_dd = eval_combo(base_e1, e2, base_e3, test_close, test_regime, test_vol_ok)
        rows.append({'param': 'EMA2', 'value': e2, 'is_sharpe': is_sr, 'oos_sharpe': oos_sr, 'is_dd': is_dd, 'oos_dd': oos_dd})
    
    # Vary EMA3
    for delta in range(-n_range, n_range + 1):
        e3 = base_e3 + delta * 10
        if e3 <= base_e2 or e3 < 20:
            continue
        is_sr, is_dd = eval_combo(base_e1, base_e2, e3, train_close, train_regime, train_vol_ok)
        oos_sr, oos_dd = eval_combo(base_e1, base_e2, e3, test_close, test_regime, test_vol_ok)
        rows.append({'param': 'EMA3', 'value': e3, 'is_sharpe': is_sr, 'oos_sharpe': oos_sr, 'is_dd': is_dd, 'oos_dd': oos_dd})
    
    return pd.DataFrame(rows)

sens_df = run_sensitivity(BEST_EMA1, BEST_EMA2, BEST_EMA3)
print(f"✅ Sensitivity analysis complete: {len(sens_df)} parameter variations tested")
print(f"   Parameters: EMA1={BEST_EMA1}, EMA2={BEST_EMA2}, EMA3={BEST_EMA3}")


In [ ]:
# Plot Sensitivity Analysis
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(f'Sensitivity Analysis — Base: EMA({BEST_EMA1},{BEST_EMA2},{BEST_EMA3})', 
             fontsize=14, fontweight='bold')

base_is_sharpe = float(pf_is.sharpe_ratio())
base_oos_sharpe = float(pf_oos.sharpe_ratio())

for col_idx, param_name in enumerate(['EMA1', 'EMA2', 'EMA3']):
    subset = sens_df[sens_df['param'] == param_name].sort_values('value')
    
    if len(subset) == 0:
        continue
    
    # IS Sharpe
    is_delta = ((subset['is_sharpe'] - base_is_sharpe) / abs(base_is_sharpe) * 100).fillna(0)
    colors = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' for x in is_delta]
    axes[0, col_idx].bar(subset['value'], is_delta, color=colors, edgecolor='black', linewidth=0.5)
    axes[0, col_idx].axhline(0, color='black', linewidth=1.5)
    base_val = [BEST_EMA1, BEST_EMA2, BEST_EMA3][col_idx]
    axes[0, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'Base={base_val}')
    axes[0, col_idx].set_title(f'{param_name} — In-Sample', fontweight='bold')
    axes[0, col_idx].set_ylabel('Sharpe Δ%')
    axes[0, col_idx].legend()
    axes[0, col_idx].grid(axis='y', alpha=0.3)
    
    # OOS Sharpe
    if not np.isnan(base_oos_sharpe) and base_oos_sharpe != 0:
        oos_delta = ((subset['oos_sharpe'] - base_oos_sharpe) / abs(base_oos_sharpe) * 100).fillna(0)
        colors_oos = ['red' if x < -10 else 'orange' if x < 0 else 'lightgreen' if x < 10 else 'darkgreen' for x in oos_delta]
        axes[1, col_idx].bar(subset['value'], oos_delta, color=colors_oos, edgecolor='black', linewidth=0.5)
        axes[1, col_idx].axhline(0, color='black', linewidth=1.5)
        axes[1, col_idx].axvline(base_val, color='blue', linestyle='--', linewidth=2, alpha=0.7, label=f'Base={base_val}')
    axes[1, col_idx].set_title(f'{param_name} — Out-of-Sample', fontweight='bold')
    axes[1, col_idx].set_ylabel('Sharpe Δ%')
    axes[1, col_idx].set_xlabel(f'{param_name} Period')
    axes[1, col_idx].legend()
    axes[1, col_idx].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

# Summary table
print("\n📋 SENSITIVITY SUMMARY")
print("=" * 80)
for param_name in ['EMA1', 'EMA2', 'EMA3']:
    subset = sens_df[sens_df['param'] == param_name]
    if len(subset) == 0:
        continue
    is_range = subset['is_sharpe'].dropna()
    oos_range = subset['oos_sharpe'].dropna()
    max_is_drop = ((is_range.min() - base_is_sharpe) / abs(base_is_sharpe) * 100) if len(is_range) > 0 else 0
    sensitivity = '⚠️ HIGH' if abs(max_is_drop) > 40 else '✅ LOW'
    print(f"   {param_name}: IS range [{is_range.min():.3f}, {is_range.max():.3f}] | "
          f"Max IS drop: {max_is_drop:.1f}% | Sensitivity: {sensitivity}")


## 15. Final Audit Cell — "Machine Learning Ridicule"

### What: A ruthless self-audit of this entire notebook, identifying everything that could have "worked by accident."
### Why: Per Meta Prompt 1: *"Assume the author is competent but possibly overconfident."* Per Carver: *"beware overconfidence!"*

This cell exists to make you uncomfortable. If nothing here makes you uncomfortable, you're not reading carefully enough.


In [ ]:
print("""
╔══════════════════════════════════════════════════════════════════════════╗
║                    🔍 FINAL AUDIT — MACHINE LEARNING RIDICULE          ║
╠══════════════════════════════════════════════════════════════════════════╣
║                                                                        ║
║  1. SURVIVORSHIP BIAS                                                  ║
║     We tested ONE asset (BTC-USD). We don't know if this works on      ║
║     the hundreds of assets we DIDN'T test. Selection of BTC-USD        ║
║     itself may be biased — it's the biggest crypto winner.             ║
║     Risk Level: SERIOUS                                                ║
║                                                                        ║
║  2. REGIME DEPENDENCY                                                  ║
║     BTC 2018-2025 contains exactly ONE major bull cycle and ONE        ║
║     major bear cycle. N=2 regimes is not a sample — it's anecdote.    ║
║     A strategy that "works" in 2 regimes may not work in 3.           ║
║     Risk Level: CRITICAL                                               ║
║                                                                        ║
║  3. PARAMETER STABILITY                                                ║
║     Even with sensitivity analysis, we optimized ~100+ combos on       ║
║     ~2000 training bars. Carver warns: "You need around 7-10 years'   ║
║     of daily data" for statistical significance at SR=0.5.             ║
║     Risk Level: SERIOUS                                                ║
║                                                                        ║
║  4. SINGLE INDICATOR FAMILY                                            ║
║     We optimized Triple EMA only. MACD-V and Donchian are defined     ║
║     but not ensembled. A true ensemble would reduce single-indicator   ║
║     overfitting risk (Giordano's Ranking Model uses 4 factors).       ║
║     Risk Level: MODERATE — easy to extend                              ║
║                                                                        ║
║  5. FEES ARE OPTIMISTIC                                                ║
║     0.1% flat fee ignores: variable spreads, slippage at market,      ║
║     overnight funding costs on CFDs/FTMO, and execution timing.       ║
║     Real FTMO costs may be 3-5x higher per round trip.                ║
║     Risk Level: SERIOUS                                                ║
║                                                                        ║
║  6. NO STOP LOSS / POSITION SIZING                                     ║
║     This notebook is 100% in or 100% out. No fractional Kelly,        ║
║     no volatility-targeted position sizing (Carver's method),          ║
║     no trailing stops. FTMO mandates max daily/total drawdown.        ║
║     Risk Level: CRITICAL for live trading                              ║
║                                                                        ║
║  7. ANTI-LOOKAHEAD IS SHIFT(1) — IS THAT ENOUGH?                     ║
║     We shift signals by 1 bar. But if data arrives with a delay       ║
║     (exchange lag, API caching), shift(1) may still be optimistic.    ║
║     For intraday strategies, shift(2) or more is safer.               ║
║     Risk Level: MODERATE                                               ║
║                                                                        ║
║  8. VOLATILITY FILTER LEAKS                                            ║
║     Rolling percentile on ATR uses a 252-day window. The FIRST       ║
║     252 bars have partial windows. We fill NaN as False (no trade),   ║
║     but the transition from NaN to valid is a regime change that      ║
║     could bias early training data.                                    ║
║     Risk Level: COSMETIC (only affects warmup)                         ║
║                                                                        ║
║  9. "WORKED BY ACCIDENT" CANDIDATES                                    ║
║     • If best EMA3 = 50 (minimum of range), we hit a boundary.       ║
║       Boundary optima are almost always overfitting artifacts.         ║
║     • If OOS Sharpe > IS Sharpe, something is VERY wrong.             ║
║       Possible data leak or lucky regime alignment.                    ║
║     • If # trades < 30, the Sharpe estimate has massive variance.     ║
║       You need ~30+ trades for any statistical meaning.               ║
║                                                                        ║
║  10. WHAT THIS NOTEBOOK DOESN'T DO                                     ║
║      • Walk-forward optimization (rolling retrain windows)             ║
║      • Monte Carlo simulation of trade sequences                       ║
║      • Correlation analysis with benchmark (beta decomposition)        ║
║      • Multi-asset portfolio construction                              ║
║      • FTMO-specific constraints (5%/10% drawdown limits)             ║
║                                                                        ║
╠══════════════════════════════════════════════════════════════════════════╣
║  VERDICT: This is a STARTING FRAMEWORK, not a trading system.          ║
║  Before risking real capital:                                          ║
║  1. Extend to multiple assets and verify cross-asset robustness        ║
║  2. Add position sizing (volatility targeting)                         ║
║  3. Implement walk-forward validation                                  ║
║  4. Add FTMO-specific risk constraints                                 ║
║  5. Paper trade for 3+ months                                          ║
╚══════════════════════════════════════════════════════════════════════════╝
""")

# Automated checks
print("\n🤖 AUTOMATED AUDIT CHECKS:")
print(f"   Best EMA1={BEST_EMA1} — {'⚠️ AT BOUNDARY' if BEST_EMA1 in [min(EMA1_RANGE), max(EMA1_RANGE)-1] else '✅ Interior'}")
print(f"   Best EMA2={BEST_EMA2} — {'⚠️ AT BOUNDARY' if BEST_EMA2 in [min(EMA2_RANGE), max(EMA2_RANGE)-1] else '✅ Interior'}")
print(f"   Best EMA3={BEST_EMA3} — {'⚠️ AT BOUNDARY' if BEST_EMA3 in [min(EMA3_RANGE), max(EMA3_RANGE)-1] else '✅ Interior'}")
print(f"   IS Sharpe:  {float(pf_is.sharpe_ratio()):.4f}")
print(f"   OOS Sharpe: {float(pf_oos.sharpe_ratio()):.4f}")

oos_sr_val = float(pf_oos.sharpe_ratio())
is_sr_val = float(pf_is.sharpe_ratio())
if oos_sr_val > is_sr_val and is_sr_val > 0:
    print(f"   ❌ OOS > IS Sharpe — SUSPICIOUS! Check for data leak.")
if pf_is.trades.count() < 30:
    print(f"   ⚠️ Only {pf_is.trades.count()} IS trades — insufficient for reliable Sharpe estimate")
if pf_oos.trades.count() < 15:
    print(f"   ⚠️ Only {pf_oos.trades.count()} OOS trades — insufficient for validation")

print(f"\n✅ Audit complete. Read the findings above carefully before deploying.")


## Appendix A: MACD-V Standalone Backtest (Optional)

### What: Quick standalone backtest of the MACD-V indicator with default parameters.
### Why: Provides a second opinion on the market. If both Triple EMA and MACD-V agree, conviction is higher.
### Assumption: Using standard 12/26/9 parameters without optimization. This is an indicator DEMO, not an optimized strategy.


In [ ]:
# MACD-V standalone demo on FULL dataset (for visualization only — not optimized)
mv_ent, mv_exi, macdv_line, sig_line = compute_macdv_signals(close, atr, 12, 26, 9)

# Apply regime filter
mv_ent = mv_ent & regime_bullish & vol_ok
mv_ent = mv_ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
mv_exi = mv_exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)

pf_macdv = vbt.Portfolio.from_signals(
    close, entries=mv_ent, exits=mv_exi,
    init_cash=100_000, fees=0.001, freq='D'
)

print(f"📊 MACD-V Demo Backtest (12/26/9, full dataset — NOT optimized)")
print(f"   Sharpe:  {pf_macdv.sharpe_ratio():.4f}")
print(f"   Return:  {pf_macdv.total_return():.2%}")
print(f"   MaxDD:   {pf_macdv.max_drawdown():.2%}")
print(f"   Trades:  {pf_macdv.trades.count()}")
print(f"\n   MACD-V value range: [{macdv_line.dropna().min():.2f}, {macdv_line.dropna().max():.2f}]")
print(f"   This is the 'invariant ruler' — values are ATR-normalized and comparable across assets.")


## Appendix B: Donchian Channel Standalone Demo


In [ ]:
# Donchian Channel demo
dc_ent, dc_exi = compute_donchian_signals(close, high, low, 20)
dc_ent = dc_ent & regime_bullish & vol_ok
dc_ent = dc_ent.shift(SIGNAL_SHIFT).fillna(False).astype(bool)
dc_exi = dc_exi.shift(SIGNAL_SHIFT).fillna(False).astype(bool)

pf_dc = vbt.Portfolio.from_signals(
    close, entries=dc_ent, exits=dc_exi,
    init_cash=100_000, fees=0.001, freq='D'
)

print(f"📊 Donchian Channel Demo (DC-20, full dataset — NOT optimized)")
print(f"   Sharpe:  {pf_dc.sharpe_ratio():.4f}")
print(f"   Return:  {pf_dc.total_return():.2%}")
print(f"   MaxDD:   {pf_dc.max_drawdown():.2%}")
print(f"   Trades:  {pf_dc.trades.count()}")


---
## End of Notebook

**Next Steps:**
1. Run each cell sequentially — never skip cells
2. Read every print statement — they ARE the audit trail
3. Check the sensitivity plots — flat plateau = good, sharp spike = bad
4. Read the Final Audit (Section 15) before making any trading decisions
5. Extend with Donchian/MACD-V grid search for a proper ensemble

*"The performance of this back-test will be amazing; too good to be true."* — Robert Carver
